# Particle Swarm Optimization (PSO)

## Introduction
- Particle Swarm Optimization (PSO) is a **population-based stochastic optimization** technique.
- Proposed by **Eberhart and Kennedy (1995)**.
- Inspired by **social behavior** of bird flocking, fish schooling, and collective swarm intelligence.
- Originally developed for **continuous nonlinear optimization**, later extended to discrete, binary, multiobjective, and constrained problems.
- PSO draws ideas from:
  - **Artificial Life** (behavioral modeling)
  - **Evolutionary Computation** (population-based search, learning)

## Swarm Intelligence Idea
- A group of particles (agents) search the space **cooperatively**.
- Each particle:
  - Has a **position** and a **velocity**.
  - Remembers its **personal best** ($pbest$).
  - Learns from the **global best** ($gbest$) among all particles.
- Particles adjust their movement by:
  - Their own best experience (cognitive learning)
  - The best performer in the swarm (social learning)

The swarm metaphor:
- Birds communicate good positions.
- Individuals adjust flying direction based on:
  - Their memory
  - Neighbors considered good

## Mathematical Formulation
- Search space is **D-dimensional**.
- There are **N particles**.
- For particle $i$ at iteration $t$:
  - Position vector $\mathbf{x}_i(t) = (x_{i1}, x_{i2}, \ldots, x_{iD})$
  - Velocity vector  $\mathbf{v}_i(t) = (v_{i1}, v_{i2}, \ldots, v_{iD})$
  - Personal best position  $\mathbf{p}_i(t)$
  - Global best position  $\mathbf{g}(t)$
$$
\Large f:\mathbb{R}^D \rightarrow \mathbb{R}
$$

$$
\Large v_{id}(t+1)=\omega v_{id}(t)
+ c_1 r_{1d}(p_{id}(t)-x_{id}(t)) 
+ c_2 r_{2d}(g_d(t)-x_{id}(t))
$$

$$
\Large x_{id}(t+1) = x_{id}(t) + v_{id}(t+1)
$$

- $\omega$ = inertia weight  
- $c_1$ = cognitive acceleration coefficient  
- $c_2$ = social acceleration coefficient  
- $r_{1d}, r_{2d} \sim U(0,1)$ random numbers  
- $g_d$ = global best position for dimension $d$

## Constriction Factor PSO (Stable Version)

Define: $\phi = c_1 + c_2$

Constriction factor:
$$
\Large K = \frac{2}{\left|2 - \phi - \sqrt{\phi^2 - 4\phi}\right|}
$$

Velocity update becomes:
$$
\Large v_{id}(t+1)= K \left[ v_{id}(t)
+ c_1 r_{1d}(p_{id}-x_{id})
+ c_2 r_{2d}(g_d-x_{id}) \right]
$$

Recommended parameters (Clerc & Kennedy):
- $c_1 = 2.05$
- $c_2 = 2.05$
- $K \approx 0.729$

## Algorithm
1. Initialize:
   - Random initial positions $x_i$
   - Random initial velocities $v_i$
   - Evaluate fitness of each particle
   - Set personal best $p_i = x_i$
   - Set global best $g$
2. Repeat for each iteration:
   - For each particle:
     - Update velocity $v_i(t+1)$
     - Update position $x_i(t+1)$
     - Clamp to search bounds
     - Evaluate fitness
     - Update personal best $p_i$
   - Update global best $g$
3. Terminate when:
   - Max iterations reached, or
   - Convergence criterion met

# Implemenation

In [21]:
import random
import math

class Particle:
    def __init__(self, dim, bounds):
        self.dim = dim
        self.position = [random.uniform(bounds[d][0], bounds[d][1]) for d in range(dim)]
        self.velocity = [0.0 for _ in range(dim)]
        self.best_position = self.position.copy()
        self.best_value = float("inf")

    def __str__(self):
        return f"Particle position {self.position} best {self.best_value}"


class PSO:
    def __init__(self, func, dim, bounds, num_particles=30, c1=2.05, c2=2.05):
        self.func = func
        self.dim = dim
        self.bounds = bounds
        self.num_particles = num_particles

        # Constriction PSO recommended
        phi = c1 + c2
        self.K = 2.0 / abs(2.0 - phi - math.sqrt(phi * phi - 4.0 * phi))

        self.c1 = c1
        self.c2 = c2

        self.particles = [Particle(dim, bounds) for _ in range(num_particles)]
        self.global_best_position = None
        self.global_best_value = float("inf")

    def evaluate(self, particle):
        value = self.func(particle.position)
        if value < particle.best_value:
            particle.best_value = value
            particle.best_position = particle.position.copy()

        if value < self.global_best_value:
            self.global_best_value = value
            self.global_best_position = particle.position.copy()

    def update_velocity(self, particle):
        for d in range(self.dim):
            r1 = random.random()
            r2 = random.random()

            cognitive = self.c1 * r1 * (particle.best_position[d] - particle.position[d])
            social = self.c2 * r2 * (self.global_best_position[d] - particle.position[d])

            particle.velocity[d] = self.K * (particle.velocity[d] + cognitive + social)

    def update_position(self, particle):
        for d in range(self.dim):
            particle.position[d] += particle.velocity[d]

            # Clamp to bounds
            low, high = self.bounds[d]
            if particle.position[d] < low:
                particle.position[d] = low
            if particle.position[d] > high:
                particle.position[d] = high

    def run(self, iterations=50, verbose=True):
        for t in range(iterations):
            for particle in self.particles:
                self.evaluate(particle)

            for particle in self.particles:
                self.update_velocity(particle)
                self.update_position(particle)

            if verbose:
                if t%100==0:
                    print(f"Iter {t} Global best value {self.global_best_value}")

        if verbose:
            print("Optimization finished")
            print(f"Best value found {self.global_best_value}")
            print(f"Best position {self.global_best_position}")

        return self.global_best_position, self.global_best_value


# Examples

In [24]:
def sphere(x):
    return sum(v * v for v in x)

bounds = [(-5, 5), (-5, 5)]
pso = PSO(func=sphere, dim=2, bounds=bounds)
pso.run(iterations=500)

Iter 0 Global best value 0.553489342569061
Iter 100 Global best value 6.541917238323284e-13
Iter 200 Global best value 7.792693223262118e-23
Iter 300 Global best value 1.6077899227161772e-30
Iter 400 Global best value 7.243881286937086e-39
Optimization finished
Best value found 1.2500414191896423e-48
Best position [-3.1022300377616527e-25, 1.074152273710639e-24]


([-3.1022300377616527e-25, 1.074152273710639e-24], 1.2500414191896423e-48)

In [25]:
def rosenbrock(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

bounds = [(-3, 3), (-3, 3)]
pso = PSO(func=rosenbrock, dim=2, bounds=bounds)
pso.run(iterations=500)

Iter 0 Global best value 14.259441706256736
Iter 100 Global best value 2.742160824821144e-05
Iter 200 Global best value 9.804943805114255e-09
Iter 300 Global best value 1.6600441151228176e-11
Iter 400 Global best value 3.9811814440828705e-13
Optimization finished
Best value found 8.279597111008676e-18
Best position [1.0000000021511164, 1.0000000044933426]


([1.0000000021511164, 1.0000000044933426], 8.279597111008676e-18)

# Portfolio Optimization Using PSO

We want to construct an optimal portfolio across multiple assets by maximizing a risk-adjusted return:

$$
\text{Score} = \mu_p - \lambda \sigma_p
$$

Where:

- $\mu_p = \sum w_i \mu_i$ → Expected portfolio return  
- $\sigma_p = \sqrt{w^T \Sigma w}$ → Portfolio volatility  
- $\lambda$ → Risk-aversion factor (higher = more cautious)

Constraints:

$$
w_i \ge 0,\quad \sum w_i = 1
$$

PSO treats each particle as a candidate weight vector $ w = (w_1, w_2, ..., w_n) $.  
The objective function outputs:

$$
\text{Minimize } -(\mu_p - \lambda \sigma_p)
$$

so that maximizing score becomes a minimization problem.

Below is the full Python example with hardcoded data and comments.


In [28]:
import numpy as np
import pandas as pd 

# ------------------------------------------------------------
# Hardcoded financial dataset
# ------------------------------------------------------------
assets = ["AAPL", "MSFT", "JPM", "XOM", "BND"]

# Expected yearly returns (μ)
expected_returns = np.array([0.12, 0.105, 0.072, 0.065, 0.03])

# Volatilities (standard deviation, σ)
vols = np.array([0.32, 0.28, 0.20, 0.25, 0.07])

df = pd.DataFrame({
    'assets':assets,
    'expected_returns':expected_returns,
    'volatility':vols
})
display(df)


# Correlation matrix (ρ)
correlation_matrix = np.array([
    [1.00, 0.82, 0.55, 0.40, 0.10],
    [0.82, 1.00, 0.50, 0.38, 0.15],
    [0.55, 0.50, 1.00, 0.35, 0.05],
    [0.40, 0.38, 0.35, 1.00, 0.00],
    [0.10, 0.15, 0.05, 0.00, 1.00]
])

df_corr = pd.DataFrame(correlation_matrix)
df_corr.columns = assets
display(df_corr) 

# ------------------------------------------------------------
# Covariance Matrix Σ = D · ρ · D
#  where D = diag(σ1, σ2, ..., σn)
# ------------------------------------------------------------
D = np.diag(vols)
cov_matrix = D @ correlation_matrix @ D    # Σ

# ------------------------------------------------------------
# Investor risk profile:
# risk_appetite ∈ [0, 1]
# 
# λ = (1 - risk_appetite) * 2
#
# Higher λ → penalizes volatility more
# ------------------------------------------------------------
risk_appetite = 0.6
lambda_risk = (1 - risk_appetite) * 2.0


# ------------------------------------------------------------
# Objective Function for PSO
#
# Given weights w:
#
# 1. Normalize: 
#       w_norm = w / Σw_i
#
# 2. Portfolio Return:
#       μ_p = Σ (w_i * μ_i)
#
# 3. Portfolio Volatility:
#       σ_p = sqrt( w^T Σ w )
#
# 4. Score:
#       Score = μ_p - λ σ_p
#
# PSO minimizes, so we return:
#       -Score
# ------------------------------------------------------------
def objective(weights):
    weights = np.array(weights)
    weights = weights / weights.sum()   # normalization ∑ w_i = 1

    # μ_p
    portfolio_return = np.sum(expected_returns * weights)

    # σ_p = sqrt(wᵀ Σ w)
    portfolio_vol = np.sqrt(weights.T @ cov_matrix @ weights)

    # Risk-adjusted objective: maximize return - λ*vol
    score = portfolio_return - lambda_risk * portfolio_vol

    return -score    # PSO minimizes


# ------------------------------------------------------------
# Bounds for weights: 0 ≤ w_i ≤ 1
# ------------------------------------------------------------
bounds = [(0, 1)] * len(assets)

# ------------------------------------------------------------
# Run PSO
# ------------------------------------------------------------
pso = PSO(func=objective, dim=len(assets), bounds=bounds)
best_w, best_value = pso.run(iterations=500)

best_w = np.array(best_w)
best_w /= best_w.sum()   # ensure final normalization



# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
print("\nOptimal Asset Allocation:")
opt_allocation = pd.DataFrame({
    'assets':assets,'best_w':best_w
})
display(opt_allocation)
print("\nInvestor-Adjusted Score:", -best_value)

,assets,expected_returns,volatility
0,AAPL,0.120,0.32
1,MSFT,0.105,0.28
2,JPM,0.072,0.20
3,XOM,0.065,0.25
4,BND,0.030,0.07


,AAPL,MSFT,JPM,XOM,BND
0,1.00,0.82,0.55,0.40,0.10
1,0.82,1.00,0.50,0.38,0.15
2,0.55,0.50,1.00,0.35,0.05
3,0.40,0.38,0.35,1.00,0.00
4,0.10,0.15,0.05,0.00,1.00


Iter 0 Global best value 0.03461765345710376
Iter 100 Global best value 0.01568641630897947
Iter 200 Global best value 0.015686416308880034
Iter 300 Global best value 0.015686416308880027
Iter 400 Global best value 0.015686416308880027
Optimization finished
Best value found 0.015686416308880027
Best position [0, 0.05313964016706201, 0.15869784820329733, 0.07896374640929368, 1.0]

Optimal Asset Allocation:


,assets,best_w
0,AAPL,0.000000
1,MSFT,0.041168
2,JPM,0.122945
3,XOM,0.061174
4,BND,0.774713



Investor-Adjusted Score: -0.015686416308880027
